# PharmaValuate - single notebook (screening aid, genuine-only detectors)
Sections: **0 setup** | **1 data + manifest** | **2 segment + split** | **3 synthetic tampers** | **4 quality gate** | **5 product ID** | **6 detectors (AnomalyDINO, PatchCore, EfficientAD)** | **7 supervised reference** | **8 calibrate + freeze** | **9 FINAL TEST (guarded)** | **10 plots**

Rules (same as AGENTS.md): never tune on test; fakes are *synthetic stress tests* only; wording = "screening only, verify manually". Every stage skips itself if its output file already exists, so a crash costs nothing. Delete the output file to force a re-run.
Dropped from the old 15 notebooks: env check (now 1 cell), OCR rules, cost/latency, MVTec sanity, perturbation set.

## 0. Setup

In [ ]:
import sys, os, json, time, glob, re, zlib, pathlib, shutil, itertools
sys.path.insert(0, os.path.abspath(".."))
from pv.config import *
import numpy as np, pandas as pd, cv2, torch, yaml
import matplotlib.pyplot as plt
from tqdm import tqdm

# ---------------- SETTINGS ----------------
STREAM = "pill"            # "pill" or "pack"  (finish pill first)
PACK_CASE = "B"            # pack only: "B" = Italian-OTC reference + degraded phone-style variants
RUN_EFFICIENTAD = True     # slowest part; set False to skip
EFFAD_SIZE, EFFAD_STEPS, SEED = "S", 15000, 0     # 15000 = first pass, 70000 = final
RUN_SUPERVISED = True
I_FROZE_EVERYTHING = False # only True after section 8 wrote configs/frozen.yaml and you committed it
# ------------------------------------------
def have(p): return pathlib.Path(p).exists()
for d in (PROC, SPLITS, SCORES, SYNTH, RES / "tables", RES / "figures", RES / "retrieval", ROOT / "configs", ROOT / "models"): d.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")
seed_everything(0)

## 1. Data audit + manifest
Pill stream reads only `c3pi/consumer` and `c3pi/reference/image/images/gallery/original`. Consumer images are mapped to products through `sample_images.xlsx`.
**First run:** the cell prints the xlsx columns and stops. Fill the four `*_COL` settings, run again.
**Important:** the C3PI *sample* you downloaded holds one drug (fluoxetine, NDC 00781-2823) and about 27 images. The manifest check below will refuse to continue until you have the real dataset (consumer images + reference gallery for hundreds of products).

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png"}          # .cr2 raw files are skipped on purpose
def imgs(root): return [p for p in pathlib.Path(root).rglob("*") if p.suffix.lower() in IMG_EXT and ".cache" not in p.parts]
NDC = re.compile(r"(\d{4,5})-(\d{3,4})")
def norm_ndc(x):
    m = NDC.search(str(x)); return f"{m.group(1).zfill(5)}-{m.group(2).zfill(4)}" if m else None

CONSUMER_XLSX_FILE_COL = None     # <- column holding the image file name / path
CONSUMER_XLSX_PRODUCT_COL = None  # <- column holding the NDC / product id
MAX_PER_PRODUCT, MAX_PRODUCTS = 20, 600

if have(PROC / f"{STREAM}_raw_manifest.csv"):
    df = pd.read_csv(PROC / f"{STREAM}_raw_manifest.csv"); print("loaded existing manifest", len(df))
elif STREAM == "pill":
    C3 = ROOT / "data/raw/c3pi"
    cx = pd.read_excel(C3 / "consumer/sample_images.xlsx"); rx = pd.read_excel(C3 / "reference/reference_sample.xlsx")
    for n, d in (("consumer xlsx", cx), ("reference xlsx", rx)): print(n, d.shape, list(d.columns)); print(d.head(5).to_string(), "\n")
    if CONSUMER_XLSX_FILE_COL is None or CONSUMER_XLSX_PRODUCT_COL is None:
        raise SystemExit("Fill CONSUMER_XLSX_FILE_COL and CONSUMER_XLSX_PRODUCT_COL from the columns printed above, then re-run this cell.")
    key = {pathlib.Path(str(f).replace("\\", "/")).name.lower(): norm_ndc(p) for f, p in zip(cx[CONSUMER_XLSX_FILE_COL], cx[CONSUMER_XLSX_PRODUCT_COL])}
    cons = imgs(C3 / "consumer"); ref = imgs(C3 / "reference/image/images/gallery/original")
    rows = [dict(path=str(p), product_id=key.get(p.name.lower()), domain="phone") for p in cons]
    rows += [dict(path=str(p), product_id=norm_ndc(p.name), domain="ref") for p in ref]
    df = pd.DataFrame(rows); n0 = len(df); df = df.dropna(subset=["product_id"])
    print(f"unmapped/dropped: {n0 - len(df)} of {n0}"); assert len(df) > 0.5 * n0, "less than half the images could be mapped to a product - check the column settings"
else:
    SRC = ROOT / ("data/raw/italian_otc" if PACK_CASE == "B" else "data/raw/mendeley")
    df = pd.DataFrame({"path": [str(p) for p in imgs(SRC)]}); df["product_id"] = [pathlib.Path(p).parent.name for p in df.path]; df["domain"] = "ref" if PACK_CASE == "B" else "phone"
    print("pack: product = parent folder name - open a few files and confirm this is right for your data")
if not have(PROC / f"{STREAM}_raw_manifest.csv"):
    df = df.groupby(["product_id", "domain"]).head(MAX_PER_PRODUCT)
    keep = df.product_id.drop_duplicates().sample(min(MAX_PRODUCTS, df.product_id.nunique()), random_state=0); df = df[df.product_id.isin(keep)].reset_index(drop=True)
print(df.domain.value_counts().to_dict(), df.product_id.nunique(), "products")

In [ ]:
def check_manifest(m, stream=STREAM):
    probs = []
    if m.product_id.nunique() < 30: probs.append(f"only {m.product_id.nunique()} products (need >= 30)")
    if stream == "pill":
        ph, rf = m[m.domain == "phone"], m[m.domain == "ref"]; both = set(ph.product_id) & set(rf.product_id)
        if len(ph) < 150: probs.append(f"only {len(ph)} phone images (need >= 150)")
        if len(both) < 10: probs.append(f"only {len(both)} products have both reference and phone images (need >= 10)")
        if (~m.product_id.astype(str).str.match(r"^\d{5}-\d{4}$")).any(): probs.append("some product ids are not NDC-style (folder names leaking in?)")
    if probs: raise SystemExit("MANIFEST CHECK FAILED - get more/correct data, do not lower thresholds:\n  - " + "\n  - ".join(probs))
    print("manifest check passed")
check_manifest(df)

In [ ]:
import imagehash
from PIL import Image
if "group_id" not in df:
    df["ph"] = [imagehash.phash(Image.open(p).convert("RGB")) for p in tqdm(df.path)]
    gid = np.empty(len(df), dtype=object)
    for pid, g in df.groupby("product_id"):
        idx = g.index.to_list(); parent = {i: i for i in idx}
        def find(i):
            while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
            return i
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                if df.ph[idx[a]] - df.ph[idx[b]] <= 6: parent[find(idx[a])] = find(idx[b])
        for i in idx: gid[i] = f"{pid}_g{find(i)}"
    df["group_id"] = gid; df["stream"] = STREAM
    df.drop(columns="ph").to_csv(PROC / f"{STREAM}_raw_manifest.csv", index=False)
    sizes = [Image.open(p).size for p in df.path.sample(min(200, len(df)), random_state=0)]
    pd.DataFrame([dict(stream=STREAM, n_images=len(df), n_products=df.product_id.nunique(), n_groups=df.group_id.nunique(),
        median_w=int(np.median([s[0] for s in sizes])), median_h=int(np.median([s[1] for s in sizes])))]).to_csv(RES / "tables" / f"audit_{STREAM}.csv", index=False)
m = pd.read_csv(PROC / f"{STREAM}_raw_manifest.csv"); print(len(m), "rows,", m.group_id.nunique(), "groups")

## 2. Segment, crop, split, leakage audit

In [ ]:
from pv.splits import make_splits, assert_no_leak
OUT = PROC / STREAM; (OUT / "clean").mkdir(parents=True, exist_ok=True); (OUT / "mask").mkdir(exist_ok=True)
def grabcut_mask(im):
    h, w = im.shape[:2]; s = 512 / max(h, w); sm = cv2.resize(im, None, fx=s, fy=s) if s < 1 else im.copy()
    mask = np.zeros(sm.shape[:2], np.uint8); H, W = sm.shape[:2]; rect = (int(W*.05), int(H*.05), int(W*.9), int(H*.9))
    bgd, fgd = np.zeros((1, 65)), np.zeros((1, 65))
    try: cv2.grabCut(cv2.cvtColor(sm, cv2.COLOR_RGB2BGR), mask, rect, bgd, fgd, 4, cv2.GC_INIT_WITH_RECT)
    except cv2.error: mask[:] = 3
    fg = ((mask == 1) | (mask == 3)).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(fg)
    if n > 1: fg = (lab == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    return cv2.resize(fg, (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
def crop_square(im, mask, size=512, margin=0.08):
    ys, xs = np.where(mask)
    if len(ys) < 50: ys, xs = np.array([0, im.shape[0]-1]), np.array([0, im.shape[1]-1]); mask = np.ones_like(mask)
    y0, y1, x0, x1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1; side = int(max(y1-y0, x1-x0)*(1+2*margin)); cy, cx = (y0+y1)//2, (x0+x1)//2; pad = side
    imp = cv2.copyMakeBorder(im, pad, pad, pad, pad, cv2.BORDER_REPLICATE); mp = cv2.copyMakeBorder(mask.astype(np.uint8), pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)
    ya, xa = cy - side//2 + pad, cx - side//2 + pad; c, cm = imp[ya:ya+side, xa:xa+side], mp[ya:ya+side, xa:xa+side]
    return cv2.resize(c, (size, size), interpolation=cv2.INTER_AREA), cv2.resize(cm, (size, size), interpolation=cv2.INTER_NEAREST) > 0
def degrade(im, rng):    # pack Case B: phone-style genuine variants from clean reference images
    im = im.astype(np.float32); M = cv2.getRotationMatrix2D((im.shape[1]/2, im.shape[0]/2), rng.uniform(-12, 12), rng.uniform(.95, 1.05)); im = cv2.warpAffine(im, M, im.shape[1::-1], borderMode=cv2.BORDER_REPLICATE)
    im = cv2.GaussianBlur(im, (0, 0), rng.uniform(0, 1.6)); im = im * rng.uniform(.7, 1.25) + rng.uniform(-18, 18)
    if rng.random() < .3:
        g = np.zeros(im.shape[:2], np.float32); cv2.circle(g, (int(rng.integers(0, im.shape[1])), int(rng.integers(0, im.shape[0]))), int(rng.integers(20, 60)), 1, -1); im += cv2.GaussianBlur(g, (0, 0), 25)[..., None] * rng.uniform(40, 110)
    im += rng.normal(0, rng.uniform(0, 5), im.shape); return np.clip(im, 0, 255).astype(np.uint8)
N_VARIANTS = 6
if not have(SPLITS / f"{STREAM}_splits.csv"):
    rows = []; rng = np.random.default_rng(0)
    for i, r in tqdm(m.iterrows(), total=len(m)):
        im = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB); c, cm = crop_square(im, grabcut_mask(im)); base = f"{i:06d}"
        cv2.imwrite(str(OUT/"clean"/f"{base}.png"), cv2.cvtColor(c, cv2.COLOR_RGB2BGR)); cv2.imwrite(str(OUT/"mask"/f"{base}.png"), cm.astype(np.uint8)*255)
        rows.append(dict(path=str(OUT/"clean"/f"{base}.png"), obj_mask=str(OUT/"mask"/f"{base}.png"), product_id=r.product_id, group_id=r.group_id, domain=r.domain, stream=STREAM))
        if STREAM == "pack" and PACK_CASE == "B":
            for k in range(N_VARIANTS):
                v = degrade(c, rng); vb = f"{base}_v{k}"
                cv2.imwrite(str(OUT/"clean"/f"{vb}.png"), cv2.cvtColor(v, cv2.COLOR_RGB2BGR)); cv2.imwrite(str(OUT/"mask"/f"{vb}.png"), cm.astype(np.uint8)*255)
                rows.append(dict(path=str(OUT/"clean"/f"{vb}.png"), obj_mask=str(OUT/"mask"/f"{vb}.png"), product_id=r.product_id, group_id=f"{r.product_id}_var{k//2}", domain="phone", stream=STREAM))
    proc = pd.DataFrame(rows)
    sp = make_splits(proc, CFG["unknown_fraction"], CFG["split_fractions"]["val"], CFG["split_fractions"]["cal"], seed=0)
    assert_no_leak(sp); sp.to_csv(SPLITS / f"{STREAM}_splits.csv", index=False)
sp = pd.read_csv(SPLITS / f"{STREAM}_splits.csv"); print(sp.split.value_counts()); print(sp.groupby(["split", "domain"]).size())

## 3. Synthetic tampers + identical JPEG re-encode of ALL images (stress-test fakes; never used to train detectors)

In [ ]:
from pv import tamper
types = CFG["tamper_types"][STREAM]; MAX_TAMPER_IMAGES = {"val": 1500, "cal": 1500, "test": 4000}
JPG = PROC / STREAM / "jpg"; MASKS = SYNTH / STREAM / "masks"; TJ = SYNTH / STREAM / "img"
for d in (JPG, MASKS, TJ): d.mkdir(parents=True, exist_ok=True)
if not have(SPLITS / f"{STREAM}_all.csv"):
    rows = []
    for r in tqdm(sp.itertuples(), total=len(sp), desc="genuine re-encode"):
        im = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB); out = JPG / (pathlib.Path(r.path).stem + ".jpg"); tamper.reencode(im, out)
        rows.append(dict(path=str(out), src_path=r.path, obj_mask=r.obj_mask, product_id=r.product_id, group_id=r.group_id, domain=r.domain, split=r.split, label=0, tamper="none", severity=0, mask_path=""))
    gen = pd.DataFrame(rows); trows = []
    for split, cap in MAX_TAMPER_IMAGES.items():
        src = gen[(gen.split == split) & (gen.domain == "phone")]; src = src.sample(min(cap, len(src)), random_state=0)
        for i, r in enumerate(tqdm(src.itertuples(), total=len(src), desc=f"tamper {split}")):
            im = cv2.cvtColor(cv2.imread(r.src_path), cv2.COLOR_BGR2RGB); om = cv2.imread(r.obj_mask, 0) > 0
            for ti, t in enumerate(types):
                sev = (i + ti) % 3 + 1; seed = zlib.crc32(f"{r.src_path}|{t}|{sev}".encode()); o, mk = tamper.apply(t, im, om, sev, seed)
                stem = f"{pathlib.Path(r.src_path).stem}__{t}{sev}"; tamper.reencode(o, TJ / f"{stem}.jpg"); cv2.imwrite(str(MASKS / f"{stem}.png"), mk.astype(np.uint8) * 255)
                trows.append(dict(path=str(TJ / f"{stem}.jpg"), src_path=r.src_path, obj_mask=r.obj_mask, product_id=r.product_id, group_id=r.group_id, domain=r.domain, split=split, label=1, tamper=t, severity=sev, mask_path=str(MASKS / f"{stem}.png")))
    pd.concat([gen, pd.DataFrame(trows)], ignore_index=True).to_csv(SPLITS / f"{STREAM}_all.csv", index=False)
all_df = pd.read_csv(SPLITS / f"{STREAM}_all.csv"); print(all_df.groupby(["split", "label"]).size())
# VISUAL CHECK: wrong-looking tampers = meaningless results
s8 = all_df[all_df.label == 1].sample(8, random_state=1); fig, ax = plt.subplots(2, 8, figsize=(24, 6))
for k, r in enumerate(s8.itertuples()):
    ax[0, k].imshow(cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)); ax[0, k].set_title(f"{r.tamper} s{r.severity}"); ax[0, k].axis("off"); ax[1, k].imshow(cv2.imread(r.mask_path, 0), cmap="gray"); ax[1, k].axis("off")
plt.tight_layout(); plt.savefig(RES / "figures" / f"tamper_examples_{STREAM}.png", dpi=100); plt.show()

## 4. Capture-quality gate (thresholds fitted on validation genuine phone images only)

In [ ]:
from pv import quality
if not have(ROOT / "configs" / f"gate_{STREAM}.json"):
    v = all_df[(all_df.split == "val") & (all_df.label == 0) & (all_df.domain == "phone")].sample(frac=1, random_state=0).head(1500)
    q = pd.DataFrame([quality.quality_metrics(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)) for p in tqdm(v.path)])
    th = quality.fit_thresholds(q, keep=0.95); json.dump(th, open(ROOT / "configs" / f"gate_{STREAM}.json", "w"), indent=1)
    print("pass rate on val genuine:", np.mean([quality.passes(r, th) for r in q.to_dict("records")]))
print(json.load(open(ROOT / "configs" / f"gate_{STREAM}.json")))

## 5. Product identification (frozen DINOv2 ViT-S/14, no training; tau set on validation only)

In [ ]:
from pv.features import DinoPatch, to_tensor, load_rgb
from pv.metrics import auroc, tpr_at_fpr
from sklearn.metrics import roc_curve
SIZE = 224
if not have(RES / "retrieval" / f"{STREAM}_pred.csv"):
    g = all_df[all_df.label == 0].copy(); dino = DinoPatch(device="cuda")
    @torch.no_grad()
    def embed(paths, rot=0, bs=32):
        out = []
        for i in range(0, len(paths), bs):
            ims = [np.ascontiguousarray(np.rot90(load_rgb(p, SIZE), rot // 90)) for p in paths[i:i+bs]]; _, cls = dino(to_tensor(ims)); out.append(torch.nn.functional.normalize(cls, dim=1).cpu())
        return torch.cat(out)
    gal = g[g.split == "gallery"]; rots = (0, 90, 180, 270) if STREAM == "pill" else (0,)
    G = torch.cat([embed(gal.path.tolist(), r) for r in rots]); Gp = np.tile(gal.product_id.to_numpy(), len(rots)); prods = np.unique(Gp); print(len(prods), "known products")
    def identify(paths):
        Q = embed(paths); S = Q @ G.T; best = np.zeros((len(paths), len(prods)), np.float32)
        for j, p in enumerate(prods): best[:, j] = S[:, torch.from_numpy(Gp == p)].max(1).values.numpy()
        o = np.argsort(-best, 1); return prods[o[:, :5]], np.take_along_axis(best, o[:, :5], 1)
    def run(df_):
        top, sim = identify(df_.path.tolist()); d = df_[["path", "product_id", "split", "domain"]].copy(); d["pred_pid"] = top[:, 0]; d["sim"] = sim[:, 0]; d["top5"] = [";".join(t) for t in top]; return d
    vk = g[(g.split == "val") & (g.domain == "phone")].drop_duplicates("src_path").head(2000); vu = g[(g.split == "unknown_val") & (g.domain == "phone")].head(1000)
    if len(vu) == 0: vu = g[g.split == "unknown_val"].head(1000)
    rk, ru = run(vk), run(vu); print(f"VAL top-1 {(rk.pred_pid == rk.product_id).mean():.3f}  top-5 {np.mean([a in t.split(';') for a, t in zip(rk.product_id, rk.top5)]):.3f}")
    y = np.r_[np.zeros(len(rk)), np.ones(len(ru))]; s = -np.r_[rk.sim, ru.sim]; print("unknown-vs-known AUROC (val):", auroc(y, s))
    f, t, th = roc_curve(y, s); tau = float(-th[np.argmax(t - f)]); json.dump(dict(tau=tau, size=SIZE), open(ROOT / "configs" / f"tau_{STREAM}.json", "w"))
    plt.hist(rk.sim, 40, alpha=.6, label="known (val)"); plt.hist(ru.sim, 40, alpha=.6, label="unknown_val"); plt.axvline(tau, c="k", ls="--", label=f"tau={tau:.2f}"); plt.legend(); plt.xlabel("best cosine similarity"); plt.savefig(RES / "figures" / f"retrieval_tau_{STREAM}.png", dpi=110); plt.show()
    rest = g[(g.split != "gallery") & (g.domain == "phone")]; pred = pd.concat([run(rest.iloc[i:i+2000]) for i in range(0, len(rest), 2000)], ignore_index=True)
    pred["pred_pid_final"] = np.where(pred.sim >= tau, pred.pred_pid, "UNKNOWN"); pred.to_csv(RES / "retrieval" / f"{STREAM}_pred.csv", index=False)
    del dino; torch.cuda.empty_cache()
print("retrieval predictions:", len(pd.read_csv(RES / "retrieval" / f"{STREAM}_pred.csv")))

## 6. Detectors (genuine-only). Tuning on a validation subsample, max 20 configs per method; no test data

In [ ]:
from pv.runner import run_method
val_sub = all_df[(all_df.split == "val") & (all_df.domain == "phone")].groupby(["product_id", "label"]).head(6)
tuning_df = pd.concat([all_df[all_df.split == "gallery"], val_sub])
full = all_df[(all_df.split.isin(["gallery", "val", "cal"])) & ((all_df.split == "gallery") | (all_df.domain == "phone"))]
ROT = (0, 90, 180, 270) if STREAM == "pill" else (0,)
tup = lambda c: {k: (list(v) if isinstance(v, tuple) else v) for k, v in c.items()}

def grid_search(cfgs, tag_name):
    out_csv = RES / "tables" / f"grid_{tag_name}_{STREAM}.csv"
    if have(out_csv): return pd.read_csv(out_csv)
    res = []
    for cfg in cfgs:
        info = run_method(cfg, STREAM, tuning_df, splits=("val",), tag="_tune")
        d = pd.read_csv(SCORES / f"{cfg['name']}__{STREAM}__val_tune.csv").dropna(subset=["score"])
        res.append(dict(cfg=json.dumps(tup(cfg)), name=cfg["name"], val_auroc=auroc(d.label, d.score), tpr_at_5fpr=tpr_at_fpr(d.label, d.score, .05), **{k: info[k] for k in ("fit_seconds", "score_seconds_val", "bank_patches_mean")}))
        print(cfg["name"], f"val AUROC {res[-1]['val_auroc']:.3f}")
    t = pd.DataFrame(res).sort_values("val_auroc", ascending=False); t.to_csv(out_csv, index=False); return t

# ---- 6a AnomalyDINO-style (frozen DINOv2 patch matching) ----
cfgs = [dict(name=f"anomalydino_c{i}", kind="dino", ratio=1.0, size=s, topfrac=tf, rotations=r) for i, (s, tf, r) in enumerate(itertools.product([336, 448], [0.005, 0.01, 0.05], [ROT, (0,)]))]
assert len(cfgs) <= CFG["tuning_budget_per_method"]
tab_dino = grid_search(cfgs, "anomalydino")
if not have(SCORES / f"anomalydino_best__{STREAM}__cal.csv"):
    best = json.loads(tab_dino.iloc[0].cfg); best["rotations"] = tuple(best["rotations"]); best["name"] = "anomalydino_best"
    print(run_method(best, STREAM, full, splits=("val", "cal"))); json.dump(tup(best), open(ROOT / "configs" / f"best_anomalydino_{STREAM}.json", "w"), indent=1)
tab_dino.head()

In [ ]:
# ---- 6b PatchCore backbone comparison (CNN baseline); bank size + timing recorded ----
cfgs = [dict(name=f"patchcore_{b}_r{int(r*100)}", kind="cnn", size=256, topfrac=0.0, rotations=ROT, out_dim=512, backbone=b, ratio=r)
        for b, r in itertools.product(["resnet18", "wide_resnet50_2", "efficientnet_b0", "convnext_tiny"], [0.01, 0.10])]
tab_pc = grid_search(cfgs, "patchcore")
if not have(SCORES / f"patchcore_best__{STREAM}__cal.csv"):
    best = json.loads(tab_pc.iloc[0].cfg); best["rotations"] = tuple(best["rotations"])
    for sd in CFG["seeds"]: run_method(dict(best, name="patchcore_best"), STREAM, full, splits=("val", "cal"), seed=sd, tag=f"_seed{sd}")
    cfg = dict(best, name="patchcore_best"); run_method(cfg, STREAM, full, splits=("val", "cal"), seed=0); json.dump(tup(cfg), open(ROOT / "configs" / f"best_patchcore_{STREAM}.json", "w"), indent=1)
# visual: detector/backbone comparison on validation
allg = pd.concat([tab_dino.assign(family="AnomalyDINO"), tab_pc.assign(family="PatchCore")]).sort_values("val_auroc")
plt.figure(figsize=(9, 0.35 * len(allg) + 1)); plt.barh(allg.name, allg.val_auroc, color=["tab:orange" if f == "AnomalyDINO" else "tab:blue" for f in allg.family]); plt.xlim(max(0.4, allg.val_auroc.min() - .05), 1); plt.xlabel("validation image AUROC (synthetic stress test)")
plt.tight_layout(); plt.savefig(RES / "figures" / f"grid_comparison_{STREAM}.png", dpi=110); plt.show(); tab_pc

In [ ]:
# ---- 6c EfficientAD (trained, anomalib==1.2.0; API changes between versions - fix imports, not the protocol) ----
name_e = f"efficientad_{EFFAD_SIZE.lower()}"; ckpt = ROOT / "models" / f"{name_e}_{STREAM}_s{SEED}.pt"
if RUN_EFFICIENTAD and not have(ckpt):
    import albumentations as A
    from anomalib.data import Folder
    from anomalib.engine import Engine
    from anomalib.models import EfficientAd
    from anomalib.models.image.efficient_ad.torch_model import EfficientAdModelSize
    from pytorch_lightning import seed_everything as pl_seed
    D = ROOT / "data" / "anomalib" / STREAM
    for s_ in ("train/good", "val/good", "val/bad"): (D / s_).mkdir(parents=True, exist_ok=True)
    aug = A.Compose([A.RandomBrightnessContrast(.25, .25, p=.8), A.HueSaturationValue(8, 20, 15, p=.6), A.GaussianBlur((3, 5), p=.3), A.Rotate(limit=10, border_mode=cv2.BORDER_REPLICATE, p=.5), A.ImageCompression(quality_range=(70, 95), p=.4)])
    gal_ = all_df[all_df.split == "gallery"]
    for i, r in enumerate(tqdm(gal_.itertuples(), total=len(gal_), desc="train/good")):
        im = cv2.imread(r.path); cv2.imwrite(str(D / "train/good" / f"{i}_orig.jpg"), im)
        for k in range(8):
            a = aug(image=cv2.cvtColor(np.ascontiguousarray(np.rot90(im, k % 4)), cv2.COLOR_BGR2RGB))["image"]; cv2.imwrite(str(D / "train/good" / f"{i}_a{k}.jpg"), cv2.cvtColor(a, cv2.COLOR_RGB2BGR))
    for i, r in enumerate(val_sub.itertuples()): shutil.copy(r.path, D / ("val/bad" if r.label else "val/good") / f"{i}.jpg")
    pl_seed(SEED)
    dm = Folder(name=f"pv_{STREAM}", root=str(D), normal_dir="train/good", abnormal_dir="val/bad", normal_test_dir="val/good", train_batch_size=1, eval_batch_size=8, num_workers=2)
    model = EfficientAd(model_size=EfficientAdModelSize.S if EFFAD_SIZE == "S" else EfficientAdModelSize.M)
    engine = Engine(max_steps=EFFAD_STEPS, max_epochs=-1, accelerator="gpu", devices=1, default_root_dir=str(RES / "logs" / f"{name_e}_{STREAM}_s{SEED}"), check_val_every_n_epoch=None, val_check_interval=min(5000, EFFAD_STEPS), enable_progress_bar=True)
    t0 = time.time(); engine.fit(model=model, datamodule=dm); train_min = (time.time() - t0) / 60; print(f"trained in {train_min:.1f} min")
    net = model.model.eval().cuda()
    @torch.no_grad()
    def eff_score(path):
        x = to_tensor([load_rgb(path, 256)]).cuda()
        try: o = net(x, normalize=False)
        except TypeError: o = net(x)
        return float(o.pred_score if hasattr(o, "pred_score") else o[0])
    for sp_ in ("val", "cal"):
        d = all_df[(all_df.split == sp_) & (all_df.domain == "phone")].copy(); d["score"] = [eff_score(p) for p in tqdm(d.path)]; d.to_csv(SCORES / f"{name_e}__{STREAM}__{sp_}_seed{SEED}.csv", index=False)
        if sp_ == "val": va = auroc(d.label, d.score); print("VAL AUROC", va)
    pd.DataFrame([dict(model=name_e, stream=STREAM, seed=SEED, steps=EFFAD_STEPS, train_minutes=train_min, val_auroc=va)]).to_csv(RES / "tables" / f"effad_{EFFAD_SIZE}_{STREAM}_s{SEED}.csv", index=False)
    torch.save(model.state_dict(), ckpt); del model, net; torch.cuda.empty_cache()
else: print("EfficientAD skipped or already trained:", ckpt)

## 7. Supervised reference (leave-one-tamper-type-out) - the only model allowed to train on tampers

In [ ]:
if RUN_SUPERVISED and not have(RES / "tables" / f"supervised_lotyo_{STREAM}_efficientnet_b0.csv"):
    from pv.sup import TamperDataset, make_model
    from torch.utils.data import DataLoader
    BACKBONE, SEEDS, EPOCHS, SPE, BS = "efficientnet_b0", [0], 8, 12800, 32
    gal_ = all_df[all_df.split == "gallery"]; val_ = all_df[(all_df.split == "val") & (all_df.domain == "phone")]
    @torch.no_grad()
    def predict(model, d_, bs=64):
        model.eval(); out = []
        for i in range(0, len(d_), bs):
            x = to_tensor([load_rgb(p, 256) for p in d_.path.iloc[i:i+bs]]).cuda()
            with torch.autocast("cuda", dtype=torch.float16): out.append(model(x).float().squeeze(1).cpu())
        return torch.cat(out).numpy()
    def train_sup(train_types, seed):
        seed_everything(seed); model = make_model(BACKBONE).cuda(); opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-4, total_steps=EPOCHS * SPE // BS); scaler = torch.cuda.amp.GradScaler()
        for ep in range(EPOCHS):
            dl = DataLoader(TamperDataset(gal_, train_types, n=SPE, seed=seed * 100 + ep), batch_size=BS, shuffle=False, num_workers=4); model.train(); tot = 0
            for x, y in dl:
                x, y = x.cuda(), y.cuda()
                with torch.autocast("cuda", dtype=torch.float16): loss = torch.nn.functional.binary_cross_entropy_with_logits(model(x).squeeze(1), y)
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
            print(f"  epoch {ep} loss {tot/len(dl):.4f}")
        return model
    rows = []
    for held in types:
        for sd in SEEDS:
            t0 = time.time(); model = train_sup([t for t in types if t != held], sd); d = val_.copy(); d["score"] = predict(model, d); gen_ = d[d.label == 0]
            res = dict(held_out=held, seed=sd, train_min=(time.time() - t0) / 60)
            for t in types: res[("unseen_" if t == held else "seen_") + t] = auroc(pd.concat([gen_, d[d.tamper == t]]).label, pd.concat([gen_, d[d.tamper == t]]).score)
            rows.append(res); print(res); del model; torch.cuda.empty_cache()
    pd.DataFrame(rows).to_csv(RES / "tables" / f"supervised_lotyo_{STREAM}_{BACKBONE}.csv", index=False)
lo = pd.read_csv(RES / "tables" / f"supervised_lotyo_{STREAM}_efficientnet_b0.csv") if have(RES / "tables" / f"supervised_lotyo_{STREAM}_efficientnet_b0.csv") else None
if lo is not None:
    seen = lo[[c for c in lo if c.startswith("seen_")]].mean(axis=1); unseen = lo[[c for c in lo if c.startswith("unseen_")]].bfill(axis=1).iloc[:, 0]
    print("mean SEEN AUROC", seen.mean(), "| mean UNSEEN AUROC", unseen.mean())
    plt.bar(["seen tampers", "unseen tamper"], [seen.mean(), unseen.mean()], color=["tab:green", "tab:red"]); plt.ylim(.4, 1); plt.ylabel("val AUROC"); plt.title("supervised classifier: seen vs unseen tamper type"); plt.savefig(RES / "figures" / f"supervised_gap_{STREAM}.png", dpi=110); plt.show()

## 8. Pick winner, Mondrian conformal calibration, FREEZE
Winner = best **validation** image AUROC across the patch-method grids (tie-break TPR@5%FPR). `alpha_gen` is chosen on calibration *synthetic* tampers - this gives **no guarantee on real counterfeits**.

In [ ]:
from pv.conformal import Mondrian, pvalues, min_cal_size, decide, fuse
grids = pd.concat([pd.read_csv(f) for f in glob.glob(str(RES / "tables" / f"grid_*_{STREAM}.csv"))]).sort_values(["val_auroc", "tpr_at_5fpr"], ascending=False)
print(grids[["name", "val_auroc", "tpr_at_5fpr"]].head(5).to_string()); winner_cfg = json.loads(grids.iloc[0].cfg); print("WINNER:", winner_cfg["name"])
family = "anomalydino" if winner_cfg["name"].startswith("anomalydino") else "patchcore"
cal = pd.read_csv(SCORES / f"{family}_best__{STREAM}__cal.csv").dropna(subset=["score"])
cal_gen = cal[(cal.label == 0) & (cal.domain == "phone")].copy(); cal_gen["stream"] = STREAM; cal_tam = cal[cal.label == 1]
print(len(cal_gen), "calibration genuine; need >=", min_cal_size(CFG["alpha_cf"]), "for alpha_cf =", CFG["alpha_cf"])
mon = Mondrian().fit(cal_gen); cells = mon.cells[(STREAM, "phone")]
p_t = pvalues(cells, cal_tam.score.to_numpy()); alpha_gen = float(np.quantile(p_t, 0.95)); alpha_gen = min(max(alpha_gen, CFG["alpha_cf"] * 2), 0.5)
print("alpha_gen =", alpha_gen, "| cal tampers with p<=alpha_cf:", (p_t <= CFG["alpha_cf"]).mean())
al = np.linspace(0.01, 0.6, 60); p_g = pvalues(cells, cal_gen.score.to_numpy())
plt.plot([(p_g <= a).mean() for a in al], [(p_t > a).mean() for a in al]); plt.xlabel("genuine flagged Inconclusive-or-worse"); plt.ylabel("tampers called Likely Genuine"); plt.title("calibration trade-off (synthetic tampers)")
plt.savefig(RES / "figures" / f"coverage_error_{STREAM}.png", dpi=110); plt.show()
fp_ = ROOT / "configs" / "frozen.yaml"; frozen = yaml.safe_load(open(fp_)) if have(fp_) else {}
frozen[STREAM] = dict(method_cfg=winner_cfg, cal_scores=[float(x) for x in np.sort(cells)], alpha_cf=CFG["alpha_cf"], alpha_gen=alpha_gen,
    tau=json.load(open(ROOT / "configs" / f"tau_{STREAM}.json"))["tau"], gate=json.load(open(ROOT / "configs" / f"gate_{STREAM}.json")))
yaml.safe_dump(frozen, open(fp_, "w")); print("FROZEN. Commit configs/frozen.yaml and git tag it before section 9.")

## 9. FINAL TEST - run once, only after freezing (set `I_FROZE_EVERYTHING = True` in section 0)

In [ ]:
assert I_FROZE_EVERYTHING, "Set I_FROZE_EVERYTHING = True in section 0 only after frozen.yaml is written and committed."
from pv.metrics import cluster_bootstrap
fz = yaml.safe_load(open(ROOT / "configs" / "frozen.yaml")); out = {}
for stream in [s for s in ("pill", "pack") if s in fz]:
    f = fz[stream]; cfg = f["method_cfg"]; cfg["rotations"] = tuple(cfg.get("rotations", (0,))); a_df = pd.read_csv(SPLITS / f"{stream}_all.csv")
    sub = a_df[(a_df.split == "gallery") | ((a_df.split == "test") & (a_df.domain == "phone"))]; run_method(cfg, stream, sub, splits=("test",), tag="_FINAL")
    ret = pd.read_csv(RES / "retrieval" / f"{stream}_pred.csv"); e2e = a_df.merge(ret[["path", "pred_pid_final"]], on="path", how="inner")
    e2e = e2e[((e2e.split == "test") | (e2e.split == "unknown_test")) & (e2e.label == 0)]
    e2e = pd.concat([a_df[a_df.split == "gallery"].assign(pred_pid_final=lambda d: d.product_id), e2e]); run_method(cfg, stream, e2e, splits=("test", "unknown_test"), pid_col="pred_pid_final", tag="_E2E_FINAL"); out[stream] = f
def load(stream, tag, sp_): return pd.read_csv(SCORES / f"{fz[stream]['method_cfg']['name']}__{stream}__{sp_}{tag}.csv")
rows = []
for stream, f in out.items():
    cal_ = np.array(f["cal_scores"]); acf, agen = f["alpha_cf"], f["alpha_gen"]
    t = load(stream, "_FINAL", "test").dropna(subset=["score"]); t["p"] = pvalues(cal_, t.score.to_numpy()); g = t[t.label == 0]
    m_, lo_, hi_ = cluster_bootstrap(g.assign(label=0), lambda d: (d.p <= acf).mean(), n=500)
    rows += [dict(test="T1", stream=stream, metric=f"false Likely-Counterfeit rate (target <= {acf})", value=(g.p <= acf).mean(), ci_lo=lo_, ci_hi=hi_),
             dict(test="T1", stream=stream, metric="Inconclusive-or-worse rate on genuine", value=(g.p <= agen).mean(), ci_lo=np.nan, ci_hi=np.nan)]
    for (tt, sv), d in t[t.label == 1].groupby(["tamper", "severity"]):
        rows.append(dict(test="T3", stream=stream, metric=f"AUROC {tt} sev{sv} (synthetic stress test)", value=auroc(pd.concat([g, d]).label, pd.concat([g, d]).score), ci_lo=np.nan, ci_hi=np.nan))
    m_, lo_, hi_ = cluster_bootstrap(t, lambda d: auroc(d.label, d.score), n=300)
    rows += [dict(test="T3", stream=stream, metric="AUROC all tampers (synthetic stress test)", value=m_, ci_lo=lo_, ci_hi=hi_), dict(test="T3", stream=stream, metric="TPR@5%FPR", value=tpr_at_fpr(t.label, t.score, .05), ci_lo=np.nan, ci_hi=np.nan)]
    u = load(stream, "_E2E_FINAL", "unknown_test"); u_rej = (u.pred_pid_final == "UNKNOWN").mean() if "pred_pid_final" in u else np.nan
    us = u.dropna(subset=["score"]); u_fc = (pvalues(cal_, us.score.to_numpy()) <= acf).sum() / max(len(u), 1)
    rows += [dict(test="T2", stream=stream, metric="Unknown-product rejection rate", value=u_rej, ci_lo=np.nan, ci_hi=np.nan), dict(test="T2", stream=stream, metric="Unknown products wrongly called Likely Counterfeit", value=u_fc, ci_lo=np.nan, ci_hi=np.nan)]
    ret = pd.read_csv(RES / "retrieval" / f"{stream}_pred.csv"); k = ret[ret.split == "test"]
    rows += [dict(test="T6", stream=stream, metric="identification top-1 (test)", value=(k.pred_pid == k.product_id).mean(), ci_lo=np.nan, ci_hi=np.nan),
             dict(test="T6", stream=stream, metric="identification top-5 (test)", value=np.mean([a in s.split(";") for a, s in zip(k.product_id, k.top5)]), ci_lo=np.nan, ci_hi=np.nan)]
final = pd.DataFrame(rows); final.to_csv(RES / "tables" / "FINAL_tests.csv", index=False); print(final.to_string())

## 10. Final plots (read from saved score files; synthetic tampers = stress test, not real counterfeits)

In [ ]:
for stream, f in out.items():
    cal_ = np.array(f["cal_scores"]); t = load(stream, "_FINAL", "test").dropna(subset=["score"]); t["p"] = pvalues(cal_, t.score.to_numpy())
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
    fpr, tpr, _ = roc_curve(t.label, t.score); ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set_title(f"{stream}: ROC genuine vs synthetic tampers"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
    ax[1].hist(t[t.label == 0].score, 50, alpha=.6, label="genuine"); ax[1].hist(t[t.label == 1].score, 50, alpha=.6, label="synthetic tamper"); ax[1].legend(); ax[1].set_title("score distributions (test)")
    g = t[t.label == 0]; per = {tt: auroc(pd.concat([g, d]).label, pd.concat([g, d]).score) for tt, d in t[t.label == 1].groupby("tamper")}
    ax[2].barh(list(per), list(per.values())); ax[2].set_xlim(.4, 1); ax[2].set_title("AUROC per tamper type (stress test)")
    plt.tight_layout(); plt.savefig(RES / "figures" / f"final_{stream}.png", dpi=110); plt.show()
    print("Decision counts on test (screening only, verify manually):"); print(pd.Series(["Likely Counterfeit" if p <= f["alpha_cf"] else "Inconclusive" if p <= f["alpha_gen"] else "Likely Genuine" for p in t.p]).groupby(t.label.map({0: "genuine", 1: "synthetic tamper"}).values).value_counts())